# 3. Subgroups and classifiers

Exploratory. Can time online and age flag low scorers (`score_pct <= 0.6`)?

- Overall logistic model: `src/modeling.py`.
- One logistic model per age-range × gender group with n ≥ 6 and at least 2 of each class: `src/subgroup_modeling.py`.
- All metrics come from stratified cross-validation.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs'
df = pd.read_csv(ROOT / 'data' / 'cleaned_dataset.csv')
import sys
sys.path.insert(0, str(ROOT / 'src'))
from common import AGE_ORDER, age_to_range
df['age_range'] = pd.Categorical(df['age'].apply(age_to_range), AGE_ORDER, ordered=True)

In [ ]:
display(Markdown('### Group sizes'))
display(pd.crosstab(df['age_range'], df['gender'], margins=True))

In [ ]:
display(Markdown('### Overall model (5-fold CV)'))
display(pd.read_csv(OUT / 'models' / 'overall_model_metrics.csv').round(3))

display(Markdown('### Subgroup models'))
m = pd.read_csv(OUT / 'models' / 'subgroup_model_metrics.csv')
pr = pd.read_csv(OUT / 'models' / 'plots' / 'subgroup_model_summary.csv')[['age_range', 'gender', 'pr_auc']]
display(m.merge(pr, on=['age_range', 'gender']).round(3))

Both kinds of model are at or near chance (ROC AUC ≈ 0.5–0.6). The 16–20 female model (AUC 0.20) is worse than chance, which is what noise looks like at n = 16. Time online and age don't predict who scores low, which matches the regression results in `02_analysis`.

In [ ]:
display(Markdown('### By gender: time online vs score (bubble size ∝ age)'))
for g in ['Female', 'Male']:
    display(Image(str(OUT / 'plots' / 'gender' / f'{g}_bubble.png'), width=600))

In [ ]:
display(Markdown('### By age × gender'))
for p in sorted((OUT / 'plots' / 'age_gender').glob('*/bubble.png')):
    display(Image(str(p), width=500))

## Reproduce
All files read here are generated by the pipeline. From the project root:

```bash
python src/run_all.py
```